## SOTA RAG with EquoAI!

In [1]:
! pip install equoai 
! pip install PyPDF2
! pip install sentence-transformers
! pip install ollama 
! pip install requests 


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from equoai import equonode as EquoNode
from sentence_transformers import SentenceTransformer
import os 
import PyPDF2
import numpy as np
from ollama import Client
import sys
import requests

/Users/matthewblack/.pyenv/versions/3.10.4/envs/david-tutorials2-3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def query_with_ollama(prompt : str, is_verbose=True) -> str:
    '''Query function using Ollama proxy'''

    # client = Client(host=os.getenv("LOCAL_SERVER_URI"))
    client = Client(host="https://ollama.newatlantis.top")

    # client = Client(host="https://pop-os.tailcff25c.ts.net/abcdefghijk")

    stream = client.chat( 
        model = "llama3.2:latest",

        # model = "mistral:7b-instruct", #This has to be added to the hardware we're using.
        messages=[{'role': 'user', 'content': prompt}],
        stream=True,
    )
    completion=""
    try:
        for chunk in stream:
            if chunk is not None:
                if is_verbose is True:
                    print(chunk['message']['content'])
                completion += chunk['message']['content']
                sys.stdout.flush()
                if chunk['message']['content'] == "<|eot_id|>":
                    stream.close() #Close the stream?
                    print("Closing the stream.")

    except Exception as e:
        print(f"{e}") 

    # return request.data["message"
    return completion 


class RAGPipeline(EquoNode):
    
    def __init__(self, model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1"):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.context = []
        self.entity_index = []
        
    def cosine(self, u: np.ndarray, v: np.ndarray) -> float:
        """
        Cosine similarity metric
        """
        return u.dot(v) / np.sqrt(u.dot(u) * v.dot(v))
    
    def process(self, query: str, documents: list[str]) -> None:
        """
        Order pieces of context by relevance to the user's query
        """
        # Generate embeddings 
        x = self.model.encode(query)
        vectors = self.model.encode(documents)
        self.context = [{"text":doc, "score":self.cosine(vectors[i], x)} for i, doc in enumerate(documents)]
        self.context= sorted(self.context, key=lambda x: x["score"], reverse=True)
        
        
    def retrieve(self, k=10) -> str:
        """
        Retrieve top-K most relevant documents
        Format: 
        Article 1: blah blah blah 
        Article 2: ...
        Article 3: ...
        """
        return "document: ".join([f'{i}: {obj["text"]} 'for i, obj in enumerate(self.context)][:k])
    
    def run(self, query: str, documents: list[str], is_optimized=False) -> str:
        """
            Handle the entire RAG, end-to-end
        """
        pipeline.process(query, documents)
        context = pipeline.retrieve()

        return query_with_ollama(f"Article: {context} \n Answer the following question using the article provided: {query}",
                 is_verbose=False)

    
def parse_pdf(file_path):
    """
    Read the local PDF file
    """
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        
        # Iterate over all the pages and extract text
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
            
        return text

In [4]:
pdf_file_path = os.path.join(os.getcwd(), 'smol.pdf')
pdf_text = parse_pdf(pdf_file_path)
document_contents = pdf_text.split(".")
project_name="esg-rag"
equo = EquoNode(project_name)
db = EquoNode(None)

In [5]:
# Model for generating Q-A embeddings

pipeline = RAGPipeline()
#Obtain embeddings and convert from ndarray to Python list. 
#Make sure that your embeddings are a list of floating point values before uploading
embeddings = pipeline.model.encode(document_contents)
embeddings = embeddings.tolist()
project_name='esg-report:power-corporation'

### Run the Basic RAG Pipeline

In [6]:
# query = 'How has the leadership of the corporation changed since Paul Desmarais stepped down?'
query = 'Who is currently in charge?'

pipeline = RAGPipeline()

pipeline.process(query, document_contents)
context = pipeline.retrieve()

answer = query_with_ollama(f"Article: {context} \n Answer the following question using the article provided: {query}",
                 is_verbose=False)

# answer = pipeline.run(query, document_contents)
print(f"Answer: {answer}")

Answer: The article doesn't explicitly state who is "currently in charge", but it does mention that André Desmarais, Deputy Chairman, was previously an executive officer of the Corporation. However, it also states that as of February 13, 2020, the Governance and Nominating Committee is now entirely composed of Directors who are not members of management, indicating a shift in leadership structure.

It's worth noting that Jeffrey Orr, President and CEO, is mentioned as an executive officer of the Corporation, but his independence status is uncertain.


### Anonymization API Demo 

#### Here, we are going to keep track of a handful of documents, anonymize their contents, 
#### and keep track of names entities along the way.
#### This allows us to do nearly anything with Generative AI, while remaining compliant in regard to data privacy and standards such as GDPR and SOC-2. 


In [7]:
from dotenv import load_dotenv
# load_dotenv()
# URI = os.getenv("URI")

URI="https://api-equo-ai.tail44cf99.ts.net"

def anonymize(text: str, entities={}, person_count=0, org_count=0, location_count=0) -> str:
    """
    Anonymize your documents using our API for data protection!
    API keys will become available for anyone signing up here:

    https://equo.ai/signup
    """
    r = requests.post(f"{URI}/protect/",
                      json={
                            "text":text,
                            "access_token":"abdefg", 
                            "entities":entities,
                            "item_counts":{
                                "person_count":person_count,
                                "org_count":org_count,
                                "location_count":location_count
                        }
    })
    return r.json()



def track_entities(doc, entities):
    """
    Keep track of previously 
    seen named entities 
    """
    for ent in list(entities.keys()):
    #     print(result["entities"][ent])
        try:
            category = entities[ent]
            doc = doc.replace(ent, category)
        except Exception as e:
            print(e)
    # print(list(result["entities"].keys()))
    return doc 

result = anonymize("Roger and Donald both work for Acme Inc. Their father worked at IBM.")

person_count = result["num_persons"] 
org_count = result["num_orgs"], 
location_count = result["num_locations"]

q2 = "David and Roger are brothers."
print(result["entities"])
entities = result["entities"]

q2 = track_entities(q2, entities)
next_result = anonymize(q2, entities, person_count, org_count, location_count)
print(next_result["text"])

{'Roger': '<PERSON 1>', 'Donald': '<PERSON 2>', 'Acme Inc.': '<ORGANIZATION 1>', 'IBM': '<ORGANIZATION 2>'}
<PERSON 3> and <PERSON 1> are brothers.


In [8]:
entities = {}
person_count = 0
org_count = 0
location_count = 0
anonymized_documents = []

documents = [
    "David and Roger are brothers. Devin is their cousin",
    "Roger and Donald both work for Acme Inc. Their father worked at IBM.",
    "Donald and David are actually best friends. Sometimes they hang out with Roger.",
    "David and Roger are brothers",
    "Donald and Roger sometimes hang out.",
    "David does not talk to his cousin"

]

# context = [obj["text"] for obj in pipeline.context]
for i, doc in enumerate(documents):
# for i, doc in enumerate(context):
# for i, doc in enumerate(document_contents):
    try:
        # Awesome 
#         print(f'Document: {doc}')
        doc = track_entities(doc, entities)
        result = anonymize(doc, entities, person_count, org_count, location_count)
        person_count = result["num_persons"] 
        org_count = result["num_orgs"], 
        location_count = result["num_locations"]
        entities = result["entities"]
        anonymized_documents.append(doc)
        print(f'Document: {doc}')
    except Exception as e:
        print(e)

Document: David and Roger are brothers. Devin is their cousin
Expecting value: line 2 column 1 (char 1)
Document: Donald and <PERSON 1> are actually best friends. Sometimes they hang out with <PERSON 2>.
Document: <PERSON 1> and <PERSON 2> are brothers
Document: <PERSON 4> and <PERSON 2> sometimes hang out.
Document: <PERSON 1> does not talk to his cousin


In [9]:
context = "".join(anonymized_documents[:])
# context = anonymized_documents[-1]
cont = pipeline.context[0]['text']

anon_context = anonymize(cont)
print(anon_context['text'])

 and <PERSON 1> from their executive roles as Co-Chief 
Executive Officers of the <ORGANIZATION 3> on <PERSON 1>, <ORGANIZATION 1> 
is now entirely composed of <ORGANIZATION 2> who are not members of management of the <ORGANIZATION 3>


In [10]:


# question = "Who sometimes hang out together?"
question = "What happened to the leadership of the company after <PERSON 1> stepped down?"
query = f"Article: {anon_context} \n Given the article above, answer the following question: {question}"
ans = query_with_ollama(query, is_verbose=False)

#The LLM is smart enough to keep track of the protected named entities if we are!
print(f"Answer from Protected Data: {ans}")

# print(pipeline.context[0]['text'])
print(ans)
# print(anon_context)

Answer from Protected Data: Based on the article, it appears that André Desmarais stepped down as Co-Chief Executive Officer of Organization 3. After he stepped down, Organization 3 is now entirely composed of Organization 2, who are not members of management of Organization 3.
Based on the article, it appears that André Desmarais stepped down as Co-Chief Executive Officer of Organization 3. After he stepped down, Organization 3 is now entirely composed of Organization 2, who are not members of management of Organization 3.


In [11]:
# print(ans)
# print(anon)
print(anon_context, type(anon_context))
# pipeline.context[0]['text']

{'text': ' and <PERSON 1> from their executive roles as Co-Chief \nExecutive Officers of the <ORGANIZATION 3> on <PERSON 1>, <ORGANIZATION 1> \nis now entirely composed of <ORGANIZATION 2> who are not members of management of the <ORGANIZATION 3>', 'num_persons': 1, 'num_orgs': 3, 'num_locations': 0, 'entities': {'André Desmarais': '<PERSON 1>', 'February\xa013, 2020': '<PERSON 1>', 'the Governance and Nominating Committee': '<ORGANIZATION 1>', 'Directors': '<ORGANIZATION 2>', 'Corporation': '<ORGANIZATION 3>'}} <class 'dict'>


## We've learned how to do two things:
#### Setup a RAG Pipeline 
#### Anonymize our Data as a next step toward compliant workflows.

In [12]:
# anonymized_documents[0]
pipeline.context[0]['text']
# # context = "".join(anonymized_documents[2:])
# anon_context = ""
# for doc in anonymized_documents:
#     if entities["Paul Desmarais"] in doc:
#         print(doc)
#         anon_context += doc 
# # context = anonymized_documents[-1]

# question = "How has the leadership changed since <PERSON 10> stepped down?"
# query = f"Article: {anon_context} \n Given the article above, answer the following question: {question}"
# ans = query_with_ollama(query, is_verbose=False)
# print(ans)

' and André Desmarais from their executive roles as Co-Chief \nExecutive Officers of the Corporation on February\xa013, 2020, the Governance and Nominating Committee \nis now entirely composed of Directors who are not members of management of the Corporation'

In [13]:
print(anonymize(pipeline.context[0]['text']))

{'text': ' and <PERSON 1> from their executive roles as Co-Chief \nExecutive Officers of the <ORGANIZATION 3> on <PERSON 1>, <ORGANIZATION 1> \nis now entirely composed of <ORGANIZATION 2> who are not members of management of the <ORGANIZATION 3>', 'num_persons': 1, 'num_orgs': 3, 'num_locations': 0, 'entities': {'André Desmarais': '<PERSON 1>', 'February\xa013, 2020': '<PERSON 1>', 'the Governance and Nominating Committee': '<ORGANIZATION 1>', 'Directors': '<ORGANIZATION 2>', 'Corporation': '<ORGANIZATION 3>'}}


In [14]:
for doc in anonymized_documents:

        print(doc)

David and Roger are brothers. Devin is their cousin
Donald and <PERSON 1> are actually best friends. Sometimes they hang out with <PERSON 2>.
<PERSON 1> and <PERSON 2> are brothers
<PERSON 4> and <PERSON 2> sometimes hang out.
<PERSON 1> does not talk to his cousin
